In [15]:
!pip install -q contractions

In [16]:
import pandas as pd
df = pd.read_csv("Suicide_Detection.csv")

In [17]:
df.head()

,Unnamed: 0,text,class
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,3,Am I weird I don't get affected by compliments...,non-suicide
2,4,Finally 2020 is almost over... So I can never ...,non-suicide
3,8,i need helpjust help me im crying so hard,suicide
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232074 entries, 0 to 232073
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  232074 non-null  int64 
 1   text        232074 non-null  object
 2   class       232074 non-null  object
dtypes: int64(1), object(2)
memory usage: 5.3+ MB


In [19]:
# preprocessing

duplicated_df = df[df.duplicated()]
duplicated_df

,Unnamed: 0,text,class


In [20]:
df.isna().sum()

Unnamed: 0    0
text          0
class         0
dtype: int64

In [21]:
df['class'].value_counts()

class
suicide        116037
non-suicide    116037
Name: count, dtype: int64

In [22]:
# data cleaning  
 
import re 
import string 
import nltk 
nltk.download('stopwords') 
nltk.download('wordnet') 
from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer 
from bs4 import BeautifulSoup 
import contractions 
 
stop = set(stopwords.words('english')) 
 
# Expanding contractions
def expand_contractions(text):
    try:
        return contractions.fix(text)
    except IndexError:
        return text  # Return original text if contractions.fix() fails
 
# Function to clean data 
def preprocess_text(text): 
 
    wl = WordNetLemmatizer() 
 
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text() 
    text = expand_contractions(text)
    emoji_clean = re.compile("[" 
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0" 
                           u"\U000024C2-\U0001F251" 
                           "]+", flags=re.UNICODE) 
    text = emoji_clean.sub(r'', text) 
    text = re.sub(r'\.(?=\S)', '. ', text)
    text = re.sub(r'http\S+', '', text)
    text = "".join([ 
        word.lower() for word in text if word not in string.punctuation 
    ])
    text = " ".join([ 
        wl.lemmatize(word) for word in text.split() if word not in stop and word.isalpha()
    ])
    return text

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [23]:
'not' in stop

True

In [24]:
'no' in stop

True

In [25]:
df['text'][0]

"Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began threatening suicide. I have tirelessly spent these paat few days talking her out of it and she keeps hesitating because she wants to believe I'll come back. I know a lot of people will threaten this in order to get their way, but what happens if she really does? What do I do and how am I supposed to handle her death on my hands? I still love my wife but I cannot deal with getting cheated on again and constantly feeling insecure. I'm worried today may be the day she does it and I hope so much it doesn't happen."

In [26]:
preprocess_text(df['text'][0])

'ex wife threatening suiciderecently left wife good cheated twice lied much decided refuse go back day ago began threatening suicide tirelessly spent paat day talking keep hesitating want believe come back know lot people threaten order get way happens really supposed handle death hand still love wife cannot deal getting cheated constantly feeling insecure worried today may day hope much happen'

In [27]:
df['text'] = df['text'].apply(preprocess_text)

C:\Users\user\AppData\Local\Temp\ipykernel_8820\3378442209.py:27: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(text, "html.parser")


In [28]:
df['text'][0]

'ex wife threatening suiciderecently left wife good cheated twice lied much decided refuse go back day ago began threatening suicide tirelessly spent paat day talking keep hesitating want believe come back know lot people threaten order get way happens really supposed handle death hand still love wife cannot deal getting cheated constantly feeling insecure worried today may day hope much happen'

In [29]:
words_len = df['text'].str.split().map(lambda x: len(x))
df_temp = df.copy()
df_temp['words length'] = words_len
df_temp

,Unnamed: 0,text,class,words length
0,2,ex wife threatening suiciderecently left wife ...,suicide,61
1,3,weird get affected compliment coming someone k...,non-suicide,13
2,4,finally almost never hear bad year ever swear ...,non-suicide,11
3,8,need helpjust help cry hard,suicide,5
4,9,losthello name adam struggling year afraid pas...,suicide,200
...,...,...,...,...
232069,348103,like rock going get anything go,non-suicide,6
232070,348106,tell many friend lonely everything deprived pr...,non-suicide,13
232071,348107,pee probably taste like salty someone drank pe...,non-suicide,9
232072,348108,usual stuff find hereim posting sympathy pity ...,suicide,143


In [30]:
from collections import Counter

words = ' '.join(df['text']).split()
counter = Counter(words)

# remove stopwords
filtered = {word:count for word,count in counter.items() if word not in stop}

# most common
most_common = sorted(filtered.items(), key=lambda x: x[1], reverse=True)

# least common
least_common = sorted(filtered.items(), key=lambda x: x[1])

print("Most common:", most_common[:20])
print("Least common:", least_common[:20])

Most common: [('like', 184488), ('want', 174310), ('know', 147857), ('feel', 136634), ('life', 128079), ('would', 124561), ('get', 120451), ('cannot', 108915), ('time', 104766), ('people', 96361), ('one', 94213), ('friend', 91084), ('even', 91020), ('year', 88140), ('going', 85332), ('really', 84298), ('day', 80804), ('thing', 80711), ('think', 75686), ('go', 73790)]
Least common: [('suiciderecently', 1), ('paat', 1), ('odoverdose', 1), ('feelingswhich', 1), ('afraidanxious', 1), ('opitunity', 1), ('encoage', 1), ('suvive', 1), ('oldhello', 1), ('painfulguns', 1), ('continuedi', 1), ('suicidaledit', 1), ('yeaputting', 1), ('voiddear', 1), ('rkikpals', 1), ('galadriel', 1), ('prestar', 1), ('aen', 1), ('matho', 1), ('mathon', 1)]


In [32]:
# Text encoding

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [33]:
x_data = df['text']

In [35]:
x_data[:5]

0    ex wife threatening suiciderecently left wife ...
1    weird get affected compliment coming someone k...
2    finally almost never hear bad year ever swear ...
3                          need helpjust help cry hard
4    losthello name adam struggling year afraid pas...
Name: text, dtype: object

In [36]:
label_encode = LabelEncoder()
y_data = label_encode.fit_transform(df['class'])

In [37]:
y_data[:5]

array([1, 0, 0, 1, 1])

In [ ]:
print(label_encode.classes_)

In [40]:
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.2, random_state=42
)

In [41]:
tfidf_vectorizer = TfidfVectorizer(max_features=15000)
tfidf_vectorizer.fit(x_train, y_train)

TfidfVectorizer(max_features=15000)

In [42]:
x_train_encoded = tfidf_vectorizer.transform(x_train)
x_test_encoded = tfidf_vectorizer.transform(x_test)

In [43]:
x_train_encoded.shape

(185659, 15000)

In [44]:
# Classification

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [45]:
# decision tree
dt_classifier = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)
dt_classifier.fit(x_train_encoded, y_train)

DecisionTreeClassifier(criterion='entropy', random_state=42)

In [46]:
y_pred = dt_classifier.predict(x_test_encoded)

In [47]:
accuracy_score(y_pred, y_test)

0.853948077130238

In [49]:
dt_classifier.score(x_test_encoded, y_test)

0.853948077130238

In [ ]:
# random forest
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(x_train_encoded, y_train)

In [ ]:
y_pred = rf_classifier.predict(x_test_encoded)

In [ ]:
accuracy_score(y_pred, y_test)

In [ ]:
rf_classifier.score(x_test_encoded, y_test)

In [ ]:
# adaboost
from sklearn.ensemble import AdaBoostClassifier

adb_classifier = AdaBoostClassifier(
    n_estimators = 100, random_state = 42
)
adb_classifier.fit(x_train_encoded, y_train)

In [ ]:
y_pred = adb_classifier.predict(x_test_encoded)

In [ ]:
accuracy_score(y_pred, y_test)

In [ ]:
# gradient boosting
from sklearn.ensemble import GradientBoostingClassifier

gb_classifier = GradientBoostingClassifier(
    n_estimators=100, random_state=42
)
gb_classifier.fit(x_train_encoded, y_train)

In [ ]:
y_pred = gb_classifier.predict(x_test_encoded)

In [ ]:
accuracy_score(y_pred, y_test)

In [ ]:
# xgboost
from xgboost import XGBRFClassifier

xgb_classifier = XGBClassifier(n_estimators = 100)
xgb_classifier.fit(x_train_encoded, y_train)

In [ ]:
y_pred = xgb_classifier.predict(x_test_encoded)

In [ ]:
accuracy_score(y_pred, y_test)

In [49]:
# testing

def predict_suicide_intent(sentence):

    # Step 1: preprocess input
    cleaned_text = preprocess_text(sentence)

    # Step 2: vectorize using trained TF-IDF
    vectorized_text = tfidf_vectorizer.transform([cleaned_text])

    # Step 3: predict
    prediction = dt_classifier.predict(vectorized_text)[0]

    # Step 4: convert label back to original class
    label = label_encode.inverse_transform([prediction])[0]

    return label

In [ ]:
sentence = "it's never too late to be great"

result = predict_suicide_intent(sentence)

print("Input:", sentence)
print("Prediction:", result)

In [71]:
# save models
import joblib

joblib.dump(dt_classifier, "model.pkl")
joblib.dump(tfidf_vectorizer, "tfidf.pkl")
joblib.dump(label_encode, "label_encoder.pkl")

print("Models saved successfully")

Models saved successfully
